# 抽取准备：分步测试

抽取过程中的若干步骤单独拿出来测试，每一步一节。
各节的工具函数之后会提取成脚本。

数据模型见 `docs/designs/graph_model.md`，入库目标是 `infra/neo4j-e08/`：与 04 的样例库（`infra/neo4j/`）数据目录分开，端口相同（bolt 7687），同一时间只起一个。

## 一、种子概念入库

种子在 `data/raw/e08-paper-knowledge/seeds/`，格式与撰写约定见同目录 `README.md`。

### 1.1 加载种子文件

每个种子文件是一条主线的共享节点，顶层只有 `nodes` 与 `relationships`：

```yaml
nodes:
  - ref: res_etth1                  # 文件内引用名；正式 id 入库时分配
    labels: [Resource, Dataset]     # 主 Label + Resource 的次级 Label
    properties:                     # 写进图的属性
      name: ETTh1
      aliases: [...]
      description: >
        ...
      url: ...                      # 仅 Resource
      note: ...                     # 可选
    sources: [...]                  # 核对所用的原始出处，不进图
    verified: true                  # 事实是否已核对，不进图
relationships:
  - {from: res_etth1, type: PART_OF, to: res_ett}   # 无 anchor，出处即种子文件
```

- 主 Label 只有 `Task`、`MethodConcept`、`Resource`、`Metric` 四类。
- 关系只在同一文件的 `ref` 之间：`SUBTYPE_OF`、`OVERLAPS_WITH`、`PART_OF`、`FOR_TASK`。
- 节点和关系都不带 `id`、`anchor`。


In [ ]:
from pathlib import Path

import pandas as pd
import yaml

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').is_file())
SEEDS = REPO / "data/raw/e08-paper-knowledge/seeds"
SEED_FILES = sorted(SEEDS.glob("*.yml"))
print([p.name for p in SEED_FILES])

工具函数（之后会提取出来作为脚本使用）

In [ ]:
def load_seed(path: Path) -> dict:
    """读一个种子文件：{"name": 文件名, "nodes": [...], "relationships": [...]}，结构与 yml 相同。"""
    raw = yaml.safe_load(path.read_text(encoding="utf-8"))
    return {"name": path.name, "nodes": raw["nodes"], "relationships": raw["relationships"]}


In [ ]:
seeds = [load_seed(p) for p in SEED_FILES]

nodes_df = pd.DataFrame([
    {"file": s["name"], "ref": n["ref"], "labels": ":".join(n["labels"]), "name": n["properties"]["name"],
     "aliases": len(n["properties"].get("aliases", [])), "note": "note" in n["properties"], "verified": n.get("verified")}
    for s in seeds for n in s["nodes"]
])
rels_df = pd.DataFrame([{"file": s["name"], **r} for s in seeds for r in s["relationships"]])

display(nodes_df.groupby(["labels", "file"]).size().unstack("file", fill_value=0))
display(rels_df.groupby(["type", "file"]).size().unstack("file", fill_value=0))


In [ ]:
nodes_df[nodes_df.verified == False]   # 待核对项


### 1.2 连接 neo4j-e08，建约束与索引

- 13 种主 Label 各建 `id` 唯一约束；
- 三个全文索引按 graph_model.md 的 Query 一节：`entity_names`（名称与 aliases）、`entity_texts`（共享节点的 description 与 note）、`statement_texts`（ClaimConcept、Issue 的陈述）。

全部 `IF NOT EXISTS`，重跑无副作用。`library_status()` 打印库里现有的节点与关系，之后每步写入后再调。

In [ ]:
import os

from neo4j import GraphDatabase, RoutingControl

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_AUTH = (os.environ.get("NEO4J_USER", "neo4j"), os.environ.get("NEO4J_PASSWORD", "password"))
NEO4J_DB = os.environ.get("NEO4J_DATABASE", "neo4j")

# 空库时查不存在的属性键会收到 UNRECOGNIZED 类提示，关掉
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, notifications_disabled_classifications=["UNRECOGNIZED"])
driver.verify_connectivity()
print(NEO4J_URI, driver.get_server_info().agent)

工具函数（之后会提取出来作为脚本使用）

In [ ]:
LABELS = ["Paper", "Contribution", "Method", "MethodConcept", "Task", "Claim", "ClaimConcept", "Issue",
          "Resource", "ResourceRecord", "Experiment", "Metric", "Observation"]   # graph_model.md 的 13 种 Node
SHARED = "Paper|Method|MethodConcept|Task|Resource|Metric"                      # 带 aliases、按名称查找的共享节点

SCHEMA = [
    *(f"CREATE CONSTRAINT {l.lower()}_id IF NOT EXISTS FOR (n:{l}) REQUIRE n.id IS UNIQUE" for l in LABELS),
    f"CREATE FULLTEXT INDEX entity_names IF NOT EXISTS FOR (n:{SHARED}) ON EACH [n.name, n.title, n.aliases]",
    f"CREATE FULLTEXT INDEX entity_texts IF NOT EXISTS FOR (n:{SHARED}) ON EACH [n.description, n.note]",
    "CREATE FULLTEXT INDEX statement_texts IF NOT EXISTS FOR (n:ClaimConcept|Issue) ON EACH [n.text, n.description, n.note]",
]

def q(cypher: str, **params) -> list[dict]:
    """只读查询的简写。"""
    records, _, _ = driver.execute_query(cypher, params, database_=NEO4J_DB, routing_=RoutingControl.READ)
    return [r.data() for r in records]

def ensure_schema():
    for stmt in SCHEMA:
        driver.execute_query(stmt, database_=NEO4J_DB)

def library_status():
    labels = q("MATCH (n) UNWIND labels(n) AS l RETURN l AS label, count(*) AS n ORDER BY label")
    rels = q("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n ORDER BY type")
    total = q("MATCH (n) RETURN count(n) AS c")[0]["c"]   # Resource 带两个 label，不能按 label 相加
    print(f"节点 {total}，关系 {sum(r['n'] for r in rels)}")
    if labels:
        display(pd.DataFrame(labels).set_index("label").T)
    if rels:
        display(pd.DataFrame(rels).set_index("type").T)

In [ ]:
ensure_schema()
display(pd.DataFrame(q("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties")))
display(pd.DataFrame(q("SHOW FULLTEXT INDEXES YIELD name, labelsOrTypes, properties, state")))
library_status()

### 1.3 种子写入

逐个种子文件：

1. **查图**：按主 Label + `name` 找节点。已存在就跳过，并取它的 `id`；不存在就交给 id 分配器取下一个。
2. **写入**：一个文件一个写事务，新节点 `CREATE`，只写 `properties`（`sources`、`verified` 不进图）；关系用 `MERGE`，已存在的不会重复建，`OVERLAPS_WITH` 这类语义对称关系不分方向。

**id 分配器**是 notebook 内的全局对象 `ids`：按主 Label 分前缀（如 `task_0001`、`res_0001`），启动时读图中每个前缀已用的最大序号，之后顺序递增。分配后写入失败只会留下空号，不会重号。

跳过的已有节点不会被种子更新；按 `name` 判断存在，节点改名后重跑会被当成新节点。

工具函数（之后会提取出来作为脚本使用）

In [ ]:
PREFIX = {"Paper": "paper", "Contribution": "contrib", "Method": "method", "MethodConcept": "mc",
          "Task": "task", "Claim": "claim", "ClaimConcept": "cc", "Issue": "issue", "Resource": "res",
          "ResourceRecord": "rr", "Experiment": "exp", "Metric": "metric", "Observation": "obs"}
SYMMETRIC = {"DIFFERS_FROM", "OVERLAPS_WITH", "CONTRADICTS"}   # graph_model.md 里的语义对称关系

class IdAllocator:
    """按主 Label 顺序分配 `<prefix>_<4 位序号>`，起点是图中该前缀已用的最大序号。"""

    def __init__(self):
        self.last = {}
        for label, prefix in PREFIX.items():
            rows = q(f"MATCH (n:{label}) WHERE n.id STARTS WITH $p RETURN max(toInteger(substring(n.id, size($p)))) AS m",
                     p=prefix + "_")
            self.last[label] = rows[0]["m"] or 0

    def new(self, label: str) -> str:
        self.last[label] += 1
        return f"{PREFIX[label]}_{self.last[label]:04d}"

def resolve_seed(seed: dict) -> tuple[dict, list]:
    """ref -> id；已在图中的取已有 id，其余新分配。返回 (ref2id, 新节点的 ref 列表)。"""
    ref2id, new_refs = {}, []
    for n in seed["nodes"]:
        main = n["labels"][0]
        hit = q(f"MATCH (n:{main} {{name: $name}}) RETURN n.id AS id", name=n["properties"]["name"])
        if hit:
            ref2id[n["ref"]] = hit[0]["id"]
        else:
            ref2id[n["ref"]] = ids.new(main)
            new_refs.append(n["ref"])
    return ref2id, new_refs

def write_seed(seed: dict, ref2id: dict, new_refs: list) -> int:
    """一个写事务：CREATE 新节点，MERGE 关系。返回新建的关系数。"""
    new = set(new_refs)

    def work(tx):
        for n in seed["nodes"]:
            if n["ref"] in new:
                labels = ":".join(n["labels"])
                tx.run(f"CREATE (n:{labels}) SET n = $props", props={"id": ref2id[n["ref"]], **n["properties"]})
        created = 0
        for r in seed["relationships"]:
            arrow = "-" if r["type"] in SYMMETRIC else "->"
            summary = tx.run(f"MATCH (a {{id: $a}}) MATCH (c {{id: $c}}) MERGE (a)-[:{r['type']}]{arrow}(c)",
                             a=ref2id[r["from"]], c=ref2id[r["to"]]).consume()
            created += summary.counters.relationships_created
        return created

    with driver.session(database=NEO4J_DB) as s:
        return s.execute_write(work)

def ingest_seed(seed: dict) -> dict:
    ref2id, new_refs = resolve_seed(seed)
    rels = write_seed(seed, ref2id, new_refs)
    print(f"{seed['name']}：新建节点 {len(new_refs)}，跳过 {len(ref2id) - len(new_refs)}；"
          f"新建关系 {rels}，已有 {len(seed['relationships']) - rels}")
    return ref2id

In [ ]:
ids = IdAllocator()
print("分配起点", {k: v for k, v in ids.last.items() if v})

ref2id = {}
for seed in seeds:
    ref2id |= ingest_seed(seed)
library_status()

In [ ]:
# 重跑同一批：应全部跳过，id 与第一次一致
for seed in seeds:
    again = ingest_seed(seed)
    assert all(ref2id[r] == i for r, i in again.items())

In [ ]:
# 抽查：ref 与分配到的 id
pd.DataFrame([{"ref": r, "id": i} for r, i in ref2id.items()]).merge(nodes_df[["ref", "name"]]).head(12)